In [1]:
"""
Extraction des cours boursiers quotidiens — couche bronze.

Récupère l'historique de dix valeurs et deux indices via yfinance,
contrôle la qualité des données et écrit un Parquet partitionné par
date d'extraction.

Principe : on ne transforme rien ici. La couche bronze conserve la
donnée telle que la source l'a fournie, avec juste de quoi tracer
d'où elle vient et quand elle est arrivée.

Usage :
    python extract_market_data.py
    python extract_market_data.py --start 2020-01-01 --out ./data
"""

import argparse
import logging
from datetime import date, datetime, timezone
from pathlib import Path

import pandas as pd
import yfinance as yf

# ---------------------------------------------------------------------
# Référentiel des instruments
# ---------------------------------------------------------------------
# Le calendrier de bourse et la devise sont portés ici parce qu'ils ne
# sont pas dans les données de cours, et qu'on en aura besoin en silver
# pour aligner les séries et convertir les montants.

INSTRUMENTS = [
    # ticker,     nom,          place,       devise, type
    ("AMZN",     "Amazon",      "NASDAQ",    "USD", "action"),
    ("TSLA",     "Tesla",       "NASDAQ",    "USD", "action"),
    ("MSFT",     "Microsoft",   "NASDAQ",    "USD", "action"),
    ("META",     "Meta",        "NASDAQ",    "USD", "action"),
    ("NVDA",     "Nvidia",      "NASDAQ",    "USD", "action"),
    ("ASML",     "ASML",        "NASDAQ",    "USD", "action"),
    ("BABA",     "Alibaba",     "NYSE",      "USD", "action"),
    ("1211.HK",  "BYD",         "HKEX",      "HKD", "action"),
    ("1810.HK",  "Xiaomi",      "HKEX",      "HKD", "action"),
    ("MC.PA",    "LVMH",        "Euronext",  "EUR", "action"),
    ("^GSPC",    "S&P 500",     "NYSE",      "USD", "indice"),
    ("^HSI",     "Hang Seng",   "HKEX",      "HKD", "indice"),
]

COLONNES_SOURCE = ["open", "high", "low", "close", "volume"]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("bronze")


# ---------------------------------------------------------------------
# Extraction
# ---------------------------------------------------------------------

def telecharger(ticker: str, debut: str, fin: str) -> pd.DataFrame:
    """Télécharge l'historique d'un instrument.

    auto_adjust=False conserve la colonne 'Adj Close' séparément :
    on veut garder le prix brut ET le prix ajusté, pas l'un à la
    place de l'autre. L'ajustement sera fait explicitement en silver.
    """
    df = yf.download(
        ticker,
        start=debut,
        end=fin,
        interval="1d",
        auto_adjust=False,
        progress=False,
    )

    if df.empty:
        log.warning("%-9s aucune donnée retournée", ticker)
        return pd.DataFrame()

    # yfinance renvoie un MultiIndex de colonnes quand on passe une liste ;
    # on l'aplatit pour garder un schéma stable quel que soit l'appel.
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df = df.rename(columns={"Date": "date", "Adj Close": "adj_close"})
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]
    if "adj_close" not in df.columns:
        df["adj_close"] = df["close"]
    return df


def enrichir(df: pd.DataFrame, meta: tuple, extrait_le: datetime) -> pd.DataFrame:
    """Ajoute les métadonnées de traçabilité. Aucune valeur n'est modifiée."""
    ticker, nom, place, devise, type_ = meta
    df = df.copy()
    df["ticker"] = ticker
    df["nom"] = nom
    df["place"] = place
    df["devise"] = devise
    df["type_instrument"] = type_
    df["source"] = "yfinance"
    df["extrait_le"] = extrait_le
    return df


# ---------------------------------------------------------------------
# Contrôles qualité
# ---------------------------------------------------------------------

def controler(df: pd.DataFrame, ticker: str) -> list[str]:
    """Contrôles déterministes. Ne corrige rien : signale.

    En bronze on ne répare pas la donnée, on documente ses défauts.
    La correction appartient à la couche silver, où elle est tracée.
    """
    alertes = []

    if df.empty:
        return [f"{ticker}: dataframe vide"]

    manquantes = df[COLONNES_SOURCE + ["adj_close"]].isna().sum()
    for col, n in manquantes.items():
        if n:
            alertes.append(f"{ticker}: {n} valeurs manquantes sur '{col}'")

    n_doublons = df.duplicated(subset=["date"]).sum()
    if n_doublons:
        alertes.append(f"{ticker}: {n_doublons} dates en doublon")

    for col in ["open", "high", "low", "close", "adj_close"]:
        n = (df[col] <= 0).sum()
        if n:
            alertes.append(f"{ticker}: {n} prix nuls ou négatifs sur '{col}'")

    # Cohérence OHLC : le haut doit dominer, le bas doit être dominé.
    incoherent = ((df["high"] < df["low"]) |
                  (df["high"] < df["open"]) |
                  (df["high"] < df["close"]) |
                  (df["low"] > df["open"]) |
                  (df["low"] > df["close"])).sum()
    if incoherent:
        alertes.append(f"{ticker}: {incoherent} lignes OHLC incohérentes")

    # Trous supérieurs à une semaine : férié long, suspension de cotation,
    # ou lacune de la source. À regarder, pas à corriger automatiquement.
    ecarts = df["date"].sort_values().diff().dt.days
    longs = (ecarts > 7).sum()
    if longs:
        alertes.append(f"{ticker}: {longs} interruptions de plus de 7 jours")

    return alertes


# ---------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------


debut = "2015-01-01"
fin = date.today().isoformat()
extrait_le = datetime.now(timezone.utc)

morceaux, toutes_alertes, echecs = [], [], []

for meta in INSTRUMENTS:
    ticker = meta[0]
    df = telecharger(ticker, debut, fin)
    if df.empty:
        echecs.append(ticker)
        continue
    toutes_alertes += controler(df, ticker)
    morceaux.append(enrichir(df, meta, extrait_le))
    print(f"{ticker:<9} {len(df):>5} lignes")

complet = pd.concat(morceaux, ignore_index=True)
complet.head()

AMZN       2941 lignes
TSLA       2941 lignes
MSFT       2941 lignes
META       2941 lignes
NVDA       2941 lignes
ASML       2941 lignes
BABA       2941 lignes
1211.HK    2881 lignes
1810.HK    2015 lignes
MC.PA      2994 lignes
^GSPC      2941 lignes
^HSI       2878 lignes


,date,adj_close,close,high,low,open,volume,ticker,nom,place,devise,type_instrument,source,extrait_le
0,2015-01-02,15.4260,15.4260,15.7375,15.3480,15.6290,55664000,AMZN,Amazon,NASDAQ,USD,action,yfinance,2026-09-15 17:41:57.027813+00:00
1,2015-01-05,15.1095,15.1095,15.4190,15.0425,15.3505,55484000,AMZN,Amazon,NASDAQ,USD,action,yfinance,2026-09-15 17:41:57.027813+00:00
2,2015-01-06,14.7645,14.7645,15.1500,14.6190,15.1120,70380000,AMZN,Amazon,NASDAQ,USD,action,yfinance,2026-09-15 17:41:57.027813+00:00
3,2015-01-07,14.9210,14.9210,15.0640,14.7665,14.8750,52806000,AMZN,Amazon,NASDAQ,USD,action,yfinance,2026-09-15 17:41:57.027813+00:00
4,2015-01-08,15.0230,15.0230,15.1570,14.8055,15.0160,61768000,AMZN,Amazon,NASDAQ,USD,action,yfinance,2026-09-15 17:41:57.027813+00:00


In [2]:
from pathlib import Path

sortie = Path("data/bronze") / f"date_extraction={extrait_le.date().isoformat()}"
sortie.mkdir(parents=True, exist_ok=True)
complet.to_parquet(sortie / "cours.parquet", index=False, compression="snappy")
print(f"{len(complet)} lignes écrites")

34296 lignes écrites


In [3]:
for a in toutes_alertes:
    print(a)

In [4]:
mc = complet[complet["ticker"] == "MC.PA"]
mc[mc["close"].isna()]

,date,adj_close,close,high,low,open,volume,ticker,nom,place,devise,type_instrument,source,extrait_le


In [5]:
import pyarrow
print(pyarrow.__version__)

23.0.1


In [6]:
verif = pd.read_parquet(sortie / "cours.parquet")
print(verif.shape, verif["ticker"].nunique())
print(verif.dtypes)

(34296, 14) 12
date                    datetime64[ms]
adj_close                      float64
close                          float64
high                           float64
low                            float64
open                           float64
volume                           int64
ticker                             str
nom                                str
place                              str
devise                             str
type_instrument                    str
source                             str
extrait_le         datetime64[us, UTC]
dtype: object
